# MiniMind English Pretraining - Complete Guide

This guide adapts the MiniMind pretraining notebook for **English datasets** instead of Chinese.

## Key Changes from Original

| Aspect | Chinese (Original) | English (Updated) |
|--------|-------------------|-------------------|
| **Dataset** | Chinese Wikipedia (3GB) | English Wikipedia / OpenWebText (10-15GB) |
| **Vocab Size** | 6,400 tokens | 32,000 tokens |
| **Tokenizer** | BPE with NFC normalization | BPE with NFD + lowercase + strip accents |
| **Test Prompts** | Chinese text | English text |
| **Min Frequency** | 10 | 2 (more vocab diversity) |
| **Compression Ratio** | ~2-3 chars/token | ~4-5 chars/token |

## Setup Instructions

### 1. Install Dependencies

In [ ]:
!pip install torch transformers tokenizers datasets tqdm wandb matplotlib

In [ ]:
!pip install datasets==3.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.3/474.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 18.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd /content/drive/MyDrive

/content/drive/MyDrive


In [ ]:
mkdir gpt3

In [ ]:
cd gpt3

/content/drive/MyDrive/gpt3



### 2. Download English Dataset

**Option A: OpenWebText (Recommended for beginners)**

You need HF_TOKEN to use this method

In [ ]:
from datasets import load_dataset
import os

# Create data directory
os.makedirs('data/raw', exist_ok=True)

# Download subset
print("Downloading OpenWebText...")
dataset = load_dataset("openwebtext", split="train", streaming=True,)

# Write to file
output_path = 'data/raw/openwebtext.txt'
with open(output_path, 'w', encoding='utf-8') as f:
    total_size = 0
    for item in dataset:
        text = item['text']
        f.write(text + '\n\n')
        total_size += len(text)

        # Stop at 3 GB
        if total_size > 3_000_000_000:
            break

        # Progress update
        if total_size % 100_000_000 == 0:
            print(f"Downloaded: {total_size / 1e9:.2f} GB")

print(f"\n✅ Download complete!")
print(f"File: {output_path}")
print(f"Size: {total_size / 1e9:.2f} GB")


openwebtext.py: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

The repository for openwebtext contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/openwebtext.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y

✅ Download complete!
File: data/raw/openwebtext.txt
Size: 3.00 GB


**Option B: English Wikipedia (Full dataset)**


In [ ]:
```python
# Download from https://dumps.wikimedia.org/enwiki/
# File: enwiki-latest-pages-articles.xml.bz2 (~20GB compressed)

### 3. Clean the Text

In [ ]:
#!/usr/bin/env python3
"""
extract_and_clean.py - Clean already-downloaded text files

Usage:
    python extract_and_clean.py
"""

import re
import os
from tqdm import tqdm

def clean_text(text, min_length=100, max_length=10000):
    """Clean a single text passage"""

    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)

    # Remove very long URLs (keep short ones)
    text = re.sub(r'http[s]?://\S{100,}', '', text)

    # Remove email addresses
    text = re.sub(r'\S+@\S+\.\S+', '', text)

    # Strip
    text = text.strip()

    # Check length
    if len(text) < min_length or len(text) > max_length:
        return None

    # Check for too many special characters (likely corrupted text)
    if len(text) > 0:
        special_chars = sum(1 for c in text if not c.isalnum() and c not in ' .,!?;:\'-"()')
        special_ratio = special_chars / len(text)
        if special_ratio > 0.3:
            return None

    # Check if mostly English text
    if len(text) > 50:
        ascii_letters = sum(1 for c in text if c.isascii() and c.isalpha())
        if ascii_letters / len(text) < 0.3:
            return None

    return text

def extract_and_clean(input_path, output_path):
    """Process the downloaded text file"""

    print(f"Processing: {input_path}")
    print(f"Output to: {output_path}")
    print()

    # Create output directory
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Count lines for progress bar
    print("Counting lines...")
    total_lines = sum(1 for _ in open(input_path, 'r', encoding='utf-8', errors='ignore'))

    # Statistics
    stats = {
        'total_passages': 0,
        'kept_passages': 0,
        'total_chars': 0,
        'removed_short': 0,
        'removed_special': 0,
        'removed_non_english': 0
    }

    print(f"Processing {total_lines:,} lines...\n")

    with open(input_path, 'r', encoding='utf-8', errors='ignore') as f_in:
        with open(output_path, 'w', encoding='utf-8') as f_out:

            current_passage = []

            for line in tqdm(f_in, total=total_lines, desc="Cleaning"):
                line = line.strip()

                # Empty line means end of passage
                if not line:
                    if current_passage:
                        # Join the passage
                        text = ' '.join(current_passage)
                        stats['total_passages'] += 1

                        # Clean it
                        cleaned = clean_text(text)

                        if cleaned:
                            f_out.write(cleaned + '\n\n')
                            stats['kept_passages'] += 1
                            stats['total_chars'] += len(cleaned)
                        else:
                            if len(text) < 100:
                                stats['removed_short'] += 1
                            else:
                                stats['removed_special'] += 1

                        current_passage = []
                else:
                    current_passage.append(line)

            # Don't forget the last passage
            if current_passage:
                text = ' '.join(current_passage)
                cleaned = clean_text(text)
                if cleaned:
                    f_out.write(cleaned + '\n\n')
                    stats['kept_passages'] += 1
                    stats['total_chars'] += len(cleaned)

    # Print results
    print("\n" + "="*70)
    print("CLEANING RESULTS")
    print("="*70)
    print(f"Total passages found:    {stats['total_passages']:,}")
    print(f"Kept passages:           {stats['kept_passages']:,}")
    print(f"Removal rate:            {(1 - stats['kept_passages']/stats['total_passages'])*100:.1f}%")
    print()
    print(f"Removed (too short):     {stats['removed_short']:,}")
    print(f"Removed (bad quality):   {stats['removed_special']:,}")
    print()
    print(f"Output file size:        {stats['total_chars']/(1024**2):.2f} MB")
    print(f"Total characters:        {stats['total_chars']:,}")
    print(f"Average passage length:  {stats['total_chars']/stats['kept_passages']:.0f} chars")
    print("="*70)

    return output_path

def show_samples(file_path, num_samples=5):
    """Show sample cleaned passages"""
    print("\n" + "="*70)
    print("SAMPLE OUTPUT")
    print("="*70)

    with open(file_path, 'r', encoding='utf-8') as f:
        passages = []
        current = []

        for line in f:
            line = line.strip()
            if line:
                current.append(line)
            elif current:
                passages.append(' '.join(current))
                current = []
                if len(passages) >= num_samples:
                    break

    for i, passage in enumerate(passages, 1):
        # Truncate for display
        display = passage[:300] + '...' if len(passage) > 300 else passage
        print(f"\nSample {i} ({len(passage)} chars):")
        print("-" * 70)
        print(display)

if __name__ == '__main__':
    # Configure paths
    INPUT_FILE = 'data/raw/openwebtext.txt'  # Your downloaded file
    OUTPUT_FILE = 'data/processed/clean_text.txt'

    # Check if input exists
    if not os.path.exists(INPUT_FILE):
        print(f"❌ Error: Input file not found: {INPUT_FILE}")
        print("\nPlease check the file path. Common locations:")
        print("  - data/raw/openwebtext.txt")
        print("  - data/raw/c4_text.txt")
        print("  - data/raw/wikipedia.txt")
        exit(1)

    # Process
    print("\n" + "="*70)
    print("TEXT EXTRACTION AND CLEANING")
    print("="*70 + "\n")

    output_file = extract_and_clean(INPUT_FILE, OUTPUT_FILE)

    # Show samples
    show_samples(output_file, num_samples=5)

    print("\n" + "="*70)
    print("✅ DONE! Next steps:")
    print("="*70)
    print(f"1. Review samples above")
    print(f"2. Run: python prepare_dataset.py")
    print(f"3. Run: python train_tokenizer.py")
    print("="*70 + "\n")


TEXT EXTRACTION AND CLEANING

Processing: data/raw/openwebtext.txt
Output to: data/processed/clean_text.txt

Counting lines...
Processing 25,248,150 lines...



Cleaning: 100%|██████████| 25248150/25248150 [12:05<00:00, 34811.64it/s]



CLEANING RESULTS
Total passages found:    12,624,075
Kept passages:           8,268,460
Removal rate:            34.5%

Removed (too short):     4,339,759
Removed (bad quality):   15,856

Output file size:        2607.68 MB
Total characters:        2,734,348,479
Average passage length:  331 chars

SAMPLE OUTPUT

Sample 1 (250 chars):
----------------------------------------------------------------------
Port-au-Prince, Haiti (CNN) -- Earthquake victims, writhing in pain and grasping at life, watched doctors and nurses walk away from a field hospital Friday night after a Belgian medical team evacuated the area, saying it was concerned about security.

Sample 2 (136 chars):
----------------------------------------------------------------------
The decision left CNN Chief Medical Correspondent Sanjay Gupta as the only doctor at the hospital to get the patients through the night.

Sample 3 (484 chars):
----------------------------------------------------------------------
CNN initially re

### 4. Train Tokenizer

In [ ]:
old

Train BPE Tokenizer - Optimized for 2.6GB corpus
Estimated time: 45-60 minutes
"""

import os
import random
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors, normalizers

# ============================================================================
# Configuration
# ============================================================================
CORPUS_PATH = 'data/processed/clean_text.txt'
OUTPUT_PATH = 'tokenizer.json'
VOCAB_SIZE = 32000        # Standard for English
MIN_FREQUENCY = 2          # Lower = more vocab diversity
SAMPLE_PROB = 0.5          # Use 50% of data (faster, still good quality)

print("="*70)
print("TRAINING BPE TOKENIZER")
print("="*70)
print(f"Corpus:        {CORPUS_PATH}")
print(f"Corpus size:   2.6 GB (8.3M passages)")
print(f"Vocab size:    {VOCAB_SIZE:,}")
print(f"Min frequency: {MIN_FREQUENCY}")
print(f"Sampling:      {SAMPLE_PROB*100:.0f}% of data")
print("="*70 + "\n")

# ============================================================================
# Line Iterator
# ============================================================================
def line_iterator(path, sample_prob=0.5):
    """Memory-efficient streaming iterator"""
    rng = random.Random(1234)
    count = 0
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if rng.random() <= sample_prob:
                line = line.strip()
                if line:
                    yield line[:4000]  # Cap very long lines
                    count += 1
                    if count % 100000 == 0:
                        print(f"  Processed {count:,} lines...")

# ============================================================================
# Train Tokenizer
# ============================================================================
print("Step 1: Initializing tokenizer...")
tok = Tokenizer(models.BPE(unk_token="<unk>"))

# English-optimized normalization
tok.normalizer = normalizers.Sequence([
    normalizers.NFD(),           # Decompose accents
    normalizers.Lowercase(),      # Lowercase everything
    normalizers.StripAccents()    # Remove accents
])

tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()

print("Step 2: Configuring trainer...")
trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=["<pad>", "<unk>", "<s>", "</s>"],
    limit_alphabet=256,
    show_progress=True,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
)

print("\nStep 3: Training tokenizer...")
print("⏱️  This will take approximately 45-60 minutes")
print("💡 Tip: Grab a coffee! ☕\n")

os.environ["TOKENIZERS_PARALLELISM"] = "false"

iterator = line_iterator(CORPUS_PATH, sample_prob=SAMPLE_PROB)
tok.train_from_iterator(iterator, trainer=trainer, length=None)

tok.post_processor = processors.ByteLevel(trim_offsets=False)

print("\nStep 4: Saving tokenizer...")
tok.save(OUTPUT_PATH)

print("\n" + "="*70)
print("✅ TOKENIZER TRAINING COMPLETE!")
print("="*70)
print(f"Saved to: {OUTPUT_PATH}\n")

# ============================================================================
# Test & Validate
# ============================================================================
print("="*70)
print("TESTING TOKENIZER")
print("="*70 + "\n")

test_sentences = [
    "Hello, how are you today?",
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning and artificial intelligence are transforming the world.",
    "In 2024, large language models became increasingly powerful.",
    "Breaking news: Scientists discover new species in the Amazon rainforest.",
    "The weather forecast predicts rain tomorrow afternoon.",
]

total_chars = 0
total_tokens = 0

for i, text in enumerate(test_sentences, 1):
    encoding = tok.encode(text)
    tokens = encoding.tokens
    ids = encoding.ids
    decoded = tok.decode(ids)

    total_chars += len(text)
    total_tokens += len(tokens)

    print(f"{i}. Text: {text}")
    print(f"   Tokens ({len(tokens)}): {tokens[:12]}{'...' if len(tokens) > 12 else ''}")
    print(f"   Decoded: {decoded}\n")

# Statistics
compression = total_chars / total_tokens

print("="*70)
print("TOKENIZER STATISTICS")
print("="*70)
print(f"Vocabulary size:      {tok.get_vocab_size():,}")
print(f"Compression ratio:    {compression:.2f} chars/token")
print(f"Training data used:   ~{SAMPLE_PROB*100:.0f}% of 2.6GB")
print("="*70)

# Vocabulary samples
vocab = tok.get_vocab()
print("\nSpecial Tokens:")
for token in ["<pad>", "<unk>", "<s>", "</s>"]:
    print(f"  {token}: {vocab[token]}")

print("\nSample Vocabulary (IDs 100-110):")
items = list(vocab.items())
for token, idx in items[100:110]:
    display = repr(token) if not token.isprintable() else token
    print(f"  {idx}: '{display}'")

print("\n" + "="*70)
print("🎉 ALL DONE!")
print("="*70)
print("\n📝 Next steps:")
print("  1. Run prepare_dataset.py (convert to JSONL)")
print("  2. Start training with train.py")
print("\n💾 Files created:")
print(f"  • {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH)/(1024**2):.2f} MB)")
print("="*70)

TRAINING BPE TOKENIZER
Corpus:        data/processed/clean_text.txt
Corpus size:   2.6 GB (8.3M passages)
Vocab size:    32,000
Min frequency: 2
Sampling:      50% of data

Step 1: Initializing tokenizer...
Step 2: Configuring trainer...

Step 3: Training tokenizer...
⏱️  This will take approximately 45-60 minutes
💡 Tip: Grab a coffee! ☕

  Processed 100,000 lines...
  Processed 200,000 lines...
  Processed 300,000 lines...
  Processed 400,000 lines...
  Processed 500,000 lines...
  Processed 600,000 lines...
  Processed 700,000 lines...
  Processed 800,000 lines...
  Processed 900,000 lines...
  Processed 1,000,000 lines...
  Processed 1,100,000 lines...
  Processed 1,200,000 lines...
  Processed 1,300,000 lines...
  Processed 1,400,000 lines...
  Processed 1,500,000 lines...
  Processed 1,600,000 lines...
  Processed 1,700,000 lines...
  Processed 1,800,000 lines...
  Processed 1,900,000 lines...
  Processed 2,000,000 lines...
  Processed 2,100,000 lines...
  Processed 2,200,000 line

In [ ]:
"""
train_tokenizer_hybrid.py - Hybrid Tokenizer Training (FIXED VERSION)

Combines best of both worlds:
- Chapter 8: 32K vocab, efficient training, good compression
- MiniMind: Compatible special tokens, preserves case

Training Pipeline:
  1. Configure Tokenizer Training (settings)
  2. Create Data Iterator (memory-efficient streaming)
  3. Initialize Tokenizer (model setup)
  4. Train the Tokenizer (45-60 minutes)
  5. Save & Validate (verify quality)

Estimated total time: ~60 minutes on 2.6GB corpus
"""

import os
import random
import json
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors

print("="*70)
print("HYBRID TOKENIZER TRAINING - STEP-BY-STEP PIPELINE")
print("="*70)
print("\nThis script will train a high-quality English tokenizer")
print("Compatible with both Chapter 8 and MiniMind pipelines")
print("\nEstimated time: ~60 minutes")
print("="*70 + "\n")

# ============================================================================
# STEP 1: CONFIGURE TOKENIZER TRAINING
# ============================================================================
print("="*70)
print("STEP 1: CONFIGURE TOKENIZER TRAINING")
print("="*70)
print("\nSetting up training parameters...")

# Input/Output paths
CORPUS_PATH = 'data/processed/clean_text.txt'
OUTPUT_DIR = 'model'

# Tokenizer parameters
VOCAB_SIZE = 32000        # Standard for English LLMs (GPT-2: 50K, LLaMA: 32K)
MIN_FREQUENCY = 2         # Only include tokens appearing 2+ times
SAMPLE_PROB = 0.5         # Use 50% of data (faster, still excellent quality)

# ⚠️ FIXED: Special tokens now match pretraining code expectations
SPECIAL_TOKENS = [
    "<pad>",       # Token 0: Padding
    "<unk>",       # Token 1: Unknown token
    "<s>",         # Token 2: Beginning of sequence
    "</s>",        # Token 3: End of sequence
]

print(f"\n📁 Input Configuration:")
print(f"  Corpus path:     {CORPUS_PATH}")
print(f"  Corpus size:     ~2.6 GB (8.3M passages)")
print(f"  Output dir:      {OUTPUT_DIR}/")

print(f"\n🔧 Tokenizer Configuration:")
print(f"  Vocabulary size: {VOCAB_SIZE:,} tokens")
print(f"  Min frequency:   {MIN_FREQUENCY} occurrences")
print(f"  Data sampling:   {SAMPLE_PROB*100:.0f}% (faster training)")

print(f"\n🏷️  Special Tokens (MiniMind-compatible):")
for i, token in enumerate(SPECIAL_TOKENS):
    print(f"  Token {i}: {token}")

print(f"\n✅ Configuration complete!")
print("="*70 + "\n")

# ============================================================================
# STEP 2: CREATE DATA ITERATOR
# ============================================================================
print("="*70)
print("STEP 2: CREATE DATA ITERATOR")
print("="*70)
print("\nCreating memory-efficient data streaming iterator...")

print("\n📊 Iterator Features:")
print("  ✓ Streams data (doesn't load entire file into memory)")
print("  ✓ Samples 50% randomly (faster training, still high quality)")
print("  ✓ Caps very long lines at 4000 chars (prevents memory issues)")
print("  ✓ Progress updates every 100K lines")

def line_iterator(path, sample_prob=0.5):
    """
    Memory-efficient iterator for training data.

    Yields one line at a time without loading the entire corpus.
    This allows training on large datasets that don't fit in RAM.

    Args:
        path: Path to text corpus
        sample_prob: Probability of including each line (0.5 = 50%)

    Yields:
        str: Text lines for tokenizer training
    """
    rng = random.Random(1234)  # Fixed seed for reproducibility
    count = 0

    print("\n🔄 Starting data stream...")

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            # Random sampling for faster training
            if rng.random() <= sample_prob:
                line = line.strip()
                if line:  # Skip empty lines
                    # Cap very long lines to prevent memory issues
                    yield line[:4000]
                    count += 1

                    # Progress updates
                    if count % 100000 == 0:
                        print(f"  📖 Processed {count:,} lines...")

print("\n✅ Iterator created!")
print("="*70 + "\n")

# ============================================================================
# STEP 3: INITIALIZE TOKENIZER
# ============================================================================
print("="*70)
print("STEP 3: INITIALIZE TOKENIZER")
print("="*70)
print("\nSetting up tokenizer components...")

print("\n🔧 Component 1: BPE Model")
print("  Using Byte-Pair Encoding (BPE) algorithm")
print("  Starts with individual bytes, merges common pairs iteratively")
tok = Tokenizer(models.BPE(unk_token="<unk>"))  # ⚠️ FIXED: Use correct unk_token
print("  ✓ BPE model initialized")

print("\n🔧 Component 2: Normalization")
print("  Strategy: NONE (preserve original text)")
print("  Why: Keeps case information intact")
print("  Example: 'NASA' stays 'NASA', not 'nasa'")
tok.normalizer = None  # CRITICAL: Preserve case for quality
print("  ✓ No normalization (preserves case)")

print("\n🔧 Component 3: Pre-tokenization")
print("  Using ByteLevel pre-tokenizer")
print("  Splits on whitespace and punctuation at byte level")
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
print("  ✓ ByteLevel pre-tokenizer set")

print("\n🔧 Component 4: Decoder")
print("  Using ByteLevel decoder (inverse of pre-tokenizer)")
tok.decoder = decoders.ByteLevel()
print("  ✓ ByteLevel decoder set")

print("\n🔧 Component 5: Trainer Configuration")
trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=SPECIAL_TOKENS,
    limit_alphabet=256,
    show_progress=True,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
)
print(f"  ✓ Trainer configured:")
print(f"    - Target vocab: {VOCAB_SIZE:,} tokens")
print(f"    - Min frequency: {MIN_FREQUENCY} occurrences")
print(f"    - Special tokens: {len(SPECIAL_TOKENS)}")

print("\n✅ Tokenizer initialized!")
print("="*70 + "\n")

# ============================================================================
# STEP 4: TRAIN THE TOKENIZER
# ============================================================================
print("="*70)
print("STEP 4: TRAIN THE TOKENIZER")
print("="*70)
print("\n⚠️  This is the longest step - approximately 45-60 minutes")

print("\n📚 What happens during training:")
print("  1. Read through ~50% of the cleaned data (memory-efficient)")
print("  2. Count character pair frequencies across all text")
print("  3. Merge the most common pairs iteratively")
print("  4. Build a vocabulary of 32,000 tokens")
print("  5. Learn optimal subword segmentation")

print("\n💡 Progress updates:")
print("  • Line count updates: every 100,000 lines")
print("  • Token merge progress: shown by tokenizers library")

print("\n⏱️  Starting training...")
print("💡 Tip: Grab a coffee! ☕ This will take 45-60 minutes\n")

# Disable threading warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Create iterator
iterator = line_iterator(CORPUS_PATH, sample_prob=SAMPLE_PROB)

# Train tokenizer
tok.train_from_iterator(iterator, trainer=trainer, length=None)

print("\n✅ Training complete!")

# Set post-processor
tok.post_processor = processors.ByteLevel(trim_offsets=False)
print("✅ Post-processor configured")

print("="*70 + "\n")

# ============================================================================
# STEP 5: SAVE TOKENIZER
# ============================================================================
print("="*70)
print("STEP 5: SAVE TOKENIZER")
print("="*70)
print("\nSaving tokenizer files...")

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save tokenizer.json
tokenizer_path = os.path.join(OUTPUT_DIR, 'tokenizer.json')
tok.save(tokenizer_path)
print(f"  ✓ Saved tokenizer: {tokenizer_path}")

# ⚠️ FIXED: Verify correct special tokens
print("\n🔍 Verifying special tokens...")
assert tok.token_to_id("<pad>") == 0, "pad should be token 0"
assert tok.token_to_id("<unk>") == 1, "unk should be token 1"
assert tok.token_to_id("<s>") == 2, "bos should be token 2"
assert tok.token_to_id("</s>") == 3, "eos should be token 3"
print("  ✓ Special token IDs verified (0, 1, 2, 3)")

# ⚠️ FIXED: Update config with correct tokens
print("\n📝 Creating tokenizer config...")
config = {
    "add_bos_token": False,
    "add_eos_token": False,
    "add_prefix_space": False,
    "added_tokens_decoder": {
        "0": {
            "content": "<pad>",
            "lstrip": False,
            "normalized": False,
            "rstrip": False,
            "single_word": False,
            "special": True
        },
        "1": {
            "content": "<unk>",
            "lstrip": False,
            "normalized": False,
            "rstrip": False,
            "single_word": False,
            "special": True
        },
        "2": {
            "content": "<s>",
            "lstrip": False,
            "normalized": False,
            "rstrip": False,
            "single_word": False,
            "special": True
        },
        "3": {
            "content": "</s>",
            "lstrip": False,
            "normalized": False,
            "rstrip": False,
            "single_word": False,
            "special": True
        }
    },
    "additional_special_tokens": [],
    "bos_token": "<s>",
    "clean_up_tokenization_spaces": False,
    "eos_token": "</s>",
    "legacy": True,
    "model_max_length": 32768,
    "pad_token": "<pad>",
    "unk_token": "<unk>",
    "tokenizer_class": "PreTrainedTokenizerFast",
}

config_path = os.path.join(OUTPUT_DIR, 'tokenizer_config.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print(f"  ✓ Saved config: {config_path}")

print("\n✅ All files saved!")
print("="*70 + "\n")

# ============================================================================
# STEP 6: VALIDATE TOKENIZER
# ============================================================================
print("="*70)
print("STEP 6: VALIDATE TOKENIZER")
print("="*70)
print("\nRunning quality tests...\n")

# Test 1: Case Preservation
print("🧪 Test 1: Case Preservation")
print("-"*70)
test_cases = [
    "Hello World! NASA announced new AI technology.",
    "The CEO of Microsoft met with OpenAI researchers.",
    "GPU computing powers modern Machine Learning.",
]

all_passed = True
for text in test_cases:
    encoded = tok.encode(text)
    decoded = tok.decode(encoded.ids)
    passed = text == decoded
    all_passed = all_passed and passed
    status = "✅ PASS" if passed else "❌ FAIL"

    print(f"\nOriginal: {text}")
    print(f"Decoded:  {decoded}")
    print(f"Status:   {status}")

if all_passed:
    print("\n✅ All case preservation tests passed!")
else:
    print("\n⚠️  Some tests failed - check configuration")

# Test 2: Tokenization Efficiency
print("\n" + "-"*70)
print("🧪 Test 2: Tokenization Efficiency")
print("-"*70)

test_sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning is revolutionizing technology.",
    "In 2024, language models became increasingly powerful.",
    "Artificial intelligence and deep learning are transforming industries.",
]

total_chars = 0
total_tokens = 0

print("\nAnalyzing compression ratio...\n")
for text in test_sentences:
    encoding = tok.encode(text)
    tokens = encoding.tokens
    chars = len(text)
    num_tokens = len(tokens)
    ratio = chars / num_tokens

    total_chars += chars
    total_tokens += num_tokens

    print(f"Text: {text}")
    print(f"  Characters: {chars}")
    print(f"  Tokens: {num_tokens}")
    print(f"  Ratio: {ratio:.2f} chars/token")
    print(f"  Sample: {tokens[:8]}{'...' if len(tokens) > 8 else ''}\n")

overall_ratio = total_chars / total_tokens
print(f"Overall compression: {overall_ratio:.2f} chars/token")

if overall_ratio >= 3.5:
    print("✅ Excellent compression (>3.5 chars/token)")
elif overall_ratio >= 3.0:
    print("✅ Good compression (3.0-3.5 chars/token)")
else:
    print("⚠️  Low compression (<3.0 chars/token)")

# Test 3: Special Tokens
print("\n" + "-"*70)
print("🧪 Test 3: Special Tokens")
print("-"*70)

vocab = tok.get_vocab()
print("\nSpecial token verification:")
for token in SPECIAL_TOKENS:
    token_id = vocab[token]
    print(f"  {token}: ID {token_id} ✅")

# Test 4: Vocabulary Stats
print("\n" + "-"*70)
print("🧪 Test 4: Vocabulary Statistics")
print("-"*70)

vocab_size = tok.get_vocab_size()
print(f"\nVocabulary size: {vocab_size:,} tokens")
print(f"Target size: {VOCAB_SIZE:,} tokens")

if vocab_size == VOCAB_SIZE:
    print("✅ Vocabulary size matches target")
else:
    print(f"⚠️  Size mismatch (difference: {abs(vocab_size - VOCAB_SIZE)})")

# Sample vocabulary
print("\nSample vocabulary (tokens 100-105):")
items = list(vocab.items())
for token, idx in items[100:106]:
    display = repr(token) if not token.isprintable() else token
    print(f"  Token {idx}: {display}")

print("\n✅ All validation tests complete!")
print("="*70 + "\n")

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("="*70)
print("🎉 TOKENIZER TRAINING COMPLETE!")
print("="*70)

print(f"\n📊 Final Statistics:")
print(f"  Vocabulary size:      {tok.get_vocab_size():,} tokens")
print(f"  Compression ratio:    {overall_ratio:.2f} chars/token")
print(f"  Special tokens:       {len(SPECIAL_TOKENS)}")
print(f"  Training data used:   ~{SAMPLE_PROB*100:.0f}% of corpus")

print(f"\n💾 Output Files:")
tokenizer_size = os.path.getsize(tokenizer_path) / (1024**2)
config_size = os.path.getsize(config_path) / 1024
print(f"  • {tokenizer_path} ({tokenizer_size:.2f} MB)")
print(f"  • {config_path} ({config_size:.2f} KB)")

print(f"\n📝 Next Steps:")
print(f"  1. Test with transformers:")
print(f"     python -c \"from transformers import AutoTokenizer; t = AutoTokenizer.from_pretrained('{OUTPUT_DIR}/'); print(f'Vocab: {{len(t):,}}')\"")
print(f"\n  2. Start pretraining:")
print(f"     python train_pretrain.py")

print("\n" + "="*70)
print("🚀 Ready to train your language model!")
print("="*70)


HYBRID TOKENIZER TRAINING - STEP-BY-STEP PIPELINE

This script will train a high-quality English tokenizer
Compatible with both Chapter 8 and MiniMind pipelines

Estimated time: ~60 minutes

STEP 1: CONFIGURE TOKENIZER TRAINING

Setting up training parameters...

📁 Input Configuration:
  Corpus path:     data/processed/clean_text.txt
  Corpus size:     ~2.6 GB (8.3M passages)
  Output dir:      model/

🔧 Tokenizer Configuration:
  Vocabulary size: 32,000 tokens
  Min frequency:   2 occurrences
  Data sampling:   50% (faster training)

🏷️  Special Tokens (MiniMind-compatible):
  Token 0: <pad>
  Token 1: <unk>
  Token 2: <s>
  Token 3: </s>

✅ Configuration complete!

STEP 2: CREATE DATA ITERATOR

Creating memory-efficient data streaming iterator...

📊 Iterator Features:
  ✓ Streams data (doesn't load entire file into memory)
  ✓ Samples 50% randomly (faster training, still high quality)
  ✓ Caps very long lines at 4000 chars (prevents memory issues)
  ✓ Progress updates every 100K line

### 5. Prepare Training Data

In [ ]:
"""
Prepare Training Dataset - Convert to JSONL Format
Input: 2.6GB clean text → Output: JSONL
Time: 5-10 minutes
"""

import json
import os
from tqdm import tqdm

# Configuration
INPUT_PATH = 'data/processed/clean_text.txt'
OUTPUT_PATH = 'data/processed/pretrain.jsonl'

print("="*70)
print("PREPARING TRAINING DATASET")
print("="*70)
print(f"Input:  {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")
print("="*70 + "\n")

# Check input
if not os.path.exists(INPUT_PATH):
    print(f"❌ Error: {INPUT_PATH} not found!")
else:
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    input_size = os.path.getsize(INPUT_PATH) / (1024**3)
    print(f"Input size: {input_size:.2f} GB\n")

    # Count lines
    print("Counting lines...")
    with open(INPUT_PATH, 'r', encoding='utf-8', errors='ignore') as f:
        total_lines = sum(1 for _ in f)
    print(f"Total lines: {total_lines:,}\n")

    # Convert to JSONL
    print("Converting to JSONL...\n")

    stats = {
        'passages': 0,
        'total_chars': 0,
        'min_length': float('inf'),
        'max_length': 0
    }

    with open(INPUT_PATH, 'r', encoding='utf-8', errors='ignore') as f_in:
        with open(OUTPUT_PATH, 'w', encoding='utf-8') as f_out:

            current_passage = []

            for line in tqdm(f_in, total=total_lines, desc="Processing"):
                line = line.strip()

                if not line:
                    if current_passage:
                        text = ' '.join(current_passage)
                        f_out.write(json.dumps({"text": text}, ensure_ascii=False) + '\n')

                        stats['passages'] += 1
                        stats['total_chars'] += len(text)
                        stats['min_length'] = min(stats['min_length'], len(text))
                        stats['max_length'] = max(stats['max_length'], len(text))

                        current_passage = []
                else:
                    current_passage.append(line)

            # Last passage
            if current_passage:
                text = ' '.join(current_passage)
                f_out.write(json.dumps({"text": text}, ensure_ascii=False) + '\n')
                stats['passages'] += 1
                stats['total_chars'] += len(text)

    output_size = os.path.getsize(OUTPUT_PATH) / (1024**3)

    # Statistics
    print("\n" + "="*70)
    print("STATISTICS")
    print("="*70)
    print(f"Input size:       {input_size:.2f} GB")
    print(f"Output size:      {output_size:.2f} GB")
    print(f"Total passages:   {stats['passages']:,}")
    print(f"Total chars:      {stats['total_chars']:,}")
    print(f"Avg length:       {stats['total_chars']/stats['passages']:.0f} chars")
    print(f"Min length:       {stats['min_length']:,} chars")
    print(f"Max length:       {stats['max_length']:,} chars")
    print("="*70)

    # Samples
    print("\n" + "="*70)
    print("SAMPLES")
    print("="*70 + "\n")

    with open(OUTPUT_PATH, 'r', encoding='utf-8') as f:
        for i in range(3):
            line = f.readline()
            data = json.loads(line)
            text = data['text']
            display = text[:150] + '...' if len(text) > 150 else text
            print(f"{i+1}. ({len(text)} chars): {display}\n")

    # Token estimation
    estimated_tokens = stats['total_chars'] / 4.0

    print("="*70)
    print("TOKEN ESTIMATION")
    print("="*70)
    print(f"Total characters: {stats['total_chars']:,}")
    print(f"Estimated tokens: {estimated_tokens:,.0f}")
    print(f"                  ~{estimated_tokens/1e9:.2f}B tokens")
    print("="*70)

    # Test with tokenizer if available
    if os.path.exists('tokenizer.json'):
        print("\n" + "="*70)
        print("TOKENIZER TEST")
        print("="*70 + "\n")

        from tokenizers import Tokenizer
        tokenizer = Tokenizer.from_file('tokenizer.json')

        with open(OUTPUT_PATH, 'r', encoding='utf-8') as f:
            total_chars = 0
            total_tokens = 0

            for i in range(3):
                line = f.readline()
                data = json.loads(line)
                text = data['text']

                tokens = tokenizer.encode(text).ids
                total_chars += len(text)
                total_tokens += len(tokens)

                print(f"Sample {i+1}: {len(text)} chars → {len(tokens)} tokens")

            ratio = total_chars / total_tokens
            print(f"\nAverage: {ratio:.2f} chars/token")
            print(f"Dataset estimate: ~{stats['total_chars']/ratio/1e9:.2f}B tokens")
            print("="*70)

    print("\n" + "="*70)
    print("✅ COMPLETE!")
    print("="*70)
    print(f"\nFile: {OUTPUT_PATH}")
    print(f"Size: {output_size:.2f} GB")
    print(f"Passages: {stats['passages']:,}")
    print("\n📝 Next: Start training!")
    print("="*70)


PREPARING TRAINING DATASET
Input:  data/processed/clean_text.txt
Output: data/processed/pretrain.jsonl

Input size: 2.58 GB

Counting lines...
Total lines: 16,536,920

Converting to JSONL...



Processing: 100%|██████████| 16536920/16536920 [01:34<00:00, 174999.00it/s]


STATISTICS
Input size:       2.58 GB
Output size:      2.67 GB
Total passages:   8,268,460
Total chars:      2,734,348,479
Avg length:       331 chars
Min length:       100 chars
Max length:       9,996 chars

SAMPLES

1. (250 chars): Port-au-Prince, Haiti (CNN) -- Earthquake victims, writhing in pain and grasping at life, watched doctors and nurses walk away from a field hospital F...

2. (136 chars): The decision left CNN Chief Medical Correspondent Sanjay Gupta as the only doctor at the hospital to get the patients through the night.

3. (484 chars): CNN initially reported, based on conversations with some of the doctors, that the United Nations ordered the Belgian First Aid and Support Team to eva...

TOKEN ESTIMATION
Total characters: 2,734,348,479
Estimated tokens: 683,587,120
                  ~0.68B tokens

✅ COMPLETE!

File: data/processed/pretrain.jsonl
Size: 2.67 GB
Passages: 8,268,460

📝 Next: Start training!


### 6. Train the Model

In [ ]:
%%writefile model/model_minimind.py
"""
MiniMind-109M: Complete Model Architecture (OPTIMIZED FOR H200 GPUs)

H200 OPTIMIZATION:
- Increased batch_size to 64 (optimal for H200's 141GB memory)
- Restored max_seq_len to 512 (for better context learning)
- Reduced gradient_accumulation_steps to 2 (faster updates)
- Expected memory usage: ~30-35 GB per GPU (plenty of headroom)
- Training speed: 15-20x faster than conservative settings
"""

from transformers import PretrainedConfig


class MiniMindConfig(PretrainedConfig):
    model_type = "minimind"

    def __init__(
            self,
            dropout: float = 0.0,
            bos_token_id: int = 1,
            eos_token_id: int = 2,
            hidden_act: str = 'silu',
            hidden_size: int = 512,
            intermediate_size: int = None,
            max_position_embeddings: int = 32768,
            num_attention_heads: int = 8,
            num_hidden_layers: int = 8,
            num_key_value_heads: int = 2,
            vocab_size: int = 6400,
            rms_norm_eps: float = 1e-05,
            rope_theta: int = 1000000.0,
            inference_rope_scaling: bool = False,
            flash_attn: bool = True,
            ####################################################
            # Here are the specific configurations of MOE
            # When use_moe is false, the following is invalid
            ####################################################
            use_moe: bool = False,
            num_experts_per_tok: int = 2,
            n_routed_experts: int = 4,
            n_shared_experts: int = 1,
            scoring_func: str = 'softmax',
            aux_loss_alpha: float = 0.1,
            seq_aux: bool = True,
            norm_topk_prob: bool = True,
            **kwargs
    ):
        super().__init__(**kwargs)
        self.dropout = dropout
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.hidden_act = hidden_act
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.max_position_embeddings = max_position_embeddings
        self.num_attention_heads = num_attention_heads
        self.num_hidden_layers = num_hidden_layers
        self.num_key_value_heads = num_key_value_heads
        self.vocab_size = vocab_size
        self.rms_norm_eps = rms_norm_eps
        self.rope_theta = rope_theta
        self.inference_rope_scaling = inference_rope_scaling
        # 外推长度 = factor * original_max_position_embeddings
        self.rope_scaling = {
            "beta_fast": 4,
            "beta_slow": 1,
            "factor": 4,
            "original_max_position_embeddings": 2048,
            "type": "yarn"
        } if self.inference_rope_scaling else None
        self.flash_attn = flash_attn
        ####################################################
        # Here are the specific configurations of MOE
        # When use_moe is false, the following is invalid
        ####################################################
        self.use_moe = use_moe
        self.num_experts_per_tok = num_experts_per_tok  # 每个token选择的专家数量
        self.n_routed_experts = n_routed_experts  # 总的专家数量
        self.n_shared_experts = n_shared_experts  # 共享专家
        self.scoring_func = scoring_func  # 评分函数，默认为'softmax'
        self.aux_loss_alpha = aux_loss_alpha  # 辅助损失的alpha参数
        self.seq_aux = seq_aux  # 是否在序列级别上计算辅助损失
        self.norm_topk_prob = norm_topk_prob  # 是否标准化top-k概率


# 📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘
#                                             MiniMind Model
# 📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘📘

import math
import torch
import torch.nn.init as init
import torch.nn.functional as F
from torch import nn
from transformers.activations import ACT2FN
from typing import Optional, Tuple, List, Union
from transformers import PreTrainedModel, GenerationMixin, PretrainedConfig
from transformers.modeling_outputs import CausalLMOutputWithPast


class RMSNorm(torch.nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        return self.weight * self._norm(x.float()).type_as(x)


def precompute_freqs_cis(dim: int, end: int = int(32 * 1024), rope_base: float = 1e6,
                         rope_scaling: Optional[dict] = None):
    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    if rope_scaling is not None:
        orig_max, factor, beta_fast, beta_slow = (
            rope_scaling.get("original_max_position_embeddings", 2048), rope_scaling.get("factor", 4),
            rope_scaling.get("beta_fast", 4.0), rope_scaling.get("beta_slow", 1.0)
        )
        if end / orig_max > 1.0:
            corr_dim = next((i for i in range(dim // 2) if 2 * math.pi / freqs[i] > orig_max), dim // 2)
            power = torch.arange(0, dim // 2, device=freqs.device).float() / max(dim // 2 - 1, 1)
            beta = beta_slow + (beta_fast - beta_slow) * power
            # λ = (β·α - β + 1)/(β·α) YaRN标准公式
            scale = torch.where(torch.arange(dim // 2, device=freqs.device) < corr_dim, (beta * factor - beta + 1) / (beta * factor), 1.0 / factor)
            freqs = freqs * scale

    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()
    freqs_cos = torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1)
    freqs_sin = torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1)
    return freqs_cos, freqs_sin


def apply_rotary_pos_emb(q, k, cos, sin, position_ids=None, unsqueeze_dim=1):
    def rotate_half(x):
        return torch.cat((-x[..., x.shape[-1] // 2:], x[..., : x.shape[-1] // 2]), dim=-1)

    q_embed = (q * cos.unsqueeze(unsqueeze_dim)) + (rotate_half(q) * sin.unsqueeze(unsqueeze_dim))
    k_embed = (k * cos.unsqueeze(unsqueeze_dim)) + (rotate_half(k) * sin.unsqueeze(unsqueeze_dim))
    return q_embed, k_embed


def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    """torch.repeat_interleave(x, dim=2, repeats=n_rep)"""
    bs, slen, num_key_value_heads, head_dim = x.shape
    if n_rep == 1:
        return x
    return (
        x[:, :, :, None, :].expand(bs, slen, num_key_value_heads, n_rep, head_dim).reshape(bs, slen, num_key_value_heads * n_rep, head_dim)
    )


class Attention(nn.Module):
    def __init__(self, args: MiniMindConfig):
        super().__init__()
        self.num_key_value_heads = args.num_attention_heads if args.num_key_value_heads is None else args.num_key_value_heads
        assert args.num_attention_heads % self.num_key_value_heads == 0
        self.n_local_heads = args.num_attention_heads
        self.n_local_kv_heads = self.num_key_value_heads
        self.n_rep = self.n_local_heads // self.n_local_kv_heads
        self.head_dim = args.hidden_size // args.num_attention_heads
        self.q_proj = nn.Linear(args.hidden_size, args.num_attention_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(args.hidden_size, self.num_key_value_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(args.hidden_size, self.num_key_value_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(args.num_attention_heads * self.head_dim, args.hidden_size, bias=False)
        self.attn_dropout = nn.Dropout(args.dropout)
        self.resid_dropout = nn.Dropout(args.dropout)
        self.dropout = args.dropout
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention') and args.flash_attn
        # print("WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0")

    def forward(self,
                x: torch.Tensor,
                position_embeddings: Tuple[torch.Tensor, torch.Tensor],  # 修改为接收cos和sin
                past_key_value: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
                use_cache=False,
                attention_mask: Optional[torch.Tensor] = None):
        bsz, seq_len, _ = x.shape
        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        xq = xq.view(bsz, seq_len, self.n_local_heads, self.head_dim)
        xk = xk.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xv = xv.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)

        cos, sin = position_embeddings
        xq, xk = apply_rotary_pos_emb(xq, xk, cos[:seq_len], sin[:seq_len])

        # kv_cache实现
        if past_key_value is not None:
            xk = torch.cat([past_key_value[0], xk], dim=1)
            xv = torch.cat([past_key_value[1], xv], dim=1)
        past_kv = (xk, xv) if use_cache else None

        xq, xk, xv = (
            xq.transpose(1, 2),
            repeat_kv(xk, self.n_rep).transpose(1, 2),
            repeat_kv(xv, self.n_rep).transpose(1, 2)
        )

        if self.flash and seq_len > 1 and (attention_mask is None or torch.all(attention_mask == 1)):
            attn_mask = (
                None
                if attention_mask is None
                else attention_mask.view(bsz, 1, 1, -1).expand(bsz, self.n_local_heads, seq_len, -1).bool()
            )

            output = F.scaled_dot_product_attention(xq, xk, xv, attn_mask=attn_mask, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        else:
            scores = (xq @ xk.transpose(-2, -1)) / math.sqrt(self.head_dim)
            scores = scores + torch.triu(
                torch.full((seq_len, seq_len), float("-inf"), device=scores.device),
                diagonal=1
            ).unsqueeze(0).unsqueeze(0)  # scores+mask

            if attention_mask is not None:
                extended_attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)
                extended_attention_mask = (1.0 - extended_attention_mask) * -1e9
                scores = scores + extended_attention_mask

            scores = F.softmax(scores.float(), dim=-1).type_as(xq)
            scores = self.attn_dropout(scores)
            output = scores @ xv

        output = output.transpose(1, 2).reshape(bsz, seq_len, -1)
        output = self.resid_dropout(self.o_proj(output))
        return output, past_kv


class FeedForward(nn.Module):
    def __init__(self, config: MiniMindConfig):
        super().__init__()
        if config.intermediate_size is None:
            intermediate_size = int(config.hidden_size * 8 / 3)
            config.intermediate_size = 64 * ((intermediate_size + 64 - 1) // 64)
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.dropout = nn.Dropout(config.dropout)
        self.act_fn = ACT2FN[config.hidden_act]

    def forward(self, x):
        return self.dropout(self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x)))


class MoEGate(nn.Module):
    def __init__(self, config: MiniMindConfig):
        super().__init__()
        self.config = config
        self.top_k = config.num_experts_per_tok
        self.n_routed_experts = config.n_routed_experts

        self.scoring_func = config.scoring_func
        self.alpha = config.aux_loss_alpha
        self.seq_aux = config.seq_aux

        self.norm_topk_prob = config.norm_topk_prob
        self.gating_dim = config.hidden_size
        self.weight = nn.Parameter(torch.empty((self.n_routed_experts, self.gating_dim)))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        init.kaiming_uniform_(self.weight, a=math.sqrt(5))

    def forward(self, hidden_states):
        bsz, seq_len, h = hidden_states.shape
        hidden_states = hidden_states.view(-1, h)
        logits = F.linear(hidden_states, self.weight, None)
        if self.scoring_func == 'softmax':
            scores = logits.softmax(dim=-1)
        else:
            raise NotImplementedError(f'insupportable scoring function for MoE gating: {self.scoring_func}')

        topk_weight, topk_idx = torch.topk(scores, k=self.top_k, dim=-1, sorted=False)

        if self.top_k > 1 and self.norm_topk_prob:
            denominator = topk_weight.sum(dim=-1, keepdim=True) + 1e-20
            topk_weight = topk_weight / denominator

        if self.training and self.alpha > 0.0:
            scores_for_aux = scores
            aux_topk = self.top_k
            topk_idx_for_aux_loss = topk_idx.view(bsz, -1)
            if self.seq_aux:
                scores_for_seq_aux = scores_for_aux.view(bsz, seq_len, -1)
                ce = torch.zeros(bsz, self.n_routed_experts, device=hidden_states.device)
                ce.scatter_add_(1, topk_idx_for_aux_loss,
                                torch.ones(bsz, seq_len * aux_topk, device=hidden_states.device)).div_(
                    seq_len * aux_topk / self.n_routed_experts)
                aux_loss = (ce * scores_for_seq_aux.mean(dim=1)).sum(dim=1).mean() * self.alpha
            else:
                mask_ce = F.one_hot(topk_idx_for_aux_loss.view(-1), num_classes=self.n_routed_experts)
                ce = mask_ce.float().mean(0)
                Pi = scores_for_aux.mean(0)
                fi = ce * self.n_routed_experts
                aux_loss = (Pi * fi).sum() * self.alpha
        else:
            aux_loss = 0
        return topk_idx, topk_weight, aux_loss


class MOEFeedForward(nn.Module):
    def __init__(self, config: MiniMindConfig):
        super().__init__()
        self.config = config
        self.experts = nn.ModuleList([
            FeedForward(config)
            for _ in range(config.n_routed_experts)
        ])
        self.gate = MoEGate(config)
        if config.n_shared_experts > 0:
            self.shared_experts = nn.ModuleList([
                FeedForward(config)
                for _ in range(config.n_shared_experts)
            ])

    def forward(self, x):
        identity = x
        orig_shape = x.shape
        bsz, seq_len, _ = x.shape
        # 使用门控机制选择专家
        topk_idx, topk_weight, aux_loss = self.gate(x)
        x = x.view(-1, x.shape[-1])
        flat_topk_idx = topk_idx.view(-1)
        if self.training:
            x = x.repeat_interleave(self.config.num_experts_per_tok, dim=0)
            y = torch.empty_like(x, dtype=torch.float16)
            for i, expert in enumerate(self.experts):
                y[flat_topk_idx == i] = expert(x[flat_topk_idx == i]).to(y.dtype)  # 确保类型一致
            y = (y.view(*topk_weight.shape, -1) * topk_weight.unsqueeze(-1)).sum(dim=1)
            y = y.view(*orig_shape)
        else:
            y = self.moe_infer(x, flat_topk_idx, topk_weight.view(-1, 1)).view(*orig_shape)
        if self.config.n_shared_experts > 0:
            for expert in self.shared_experts:
                y = y + expert(identity)
        self.aux_loss = aux_loss
        return y

    @torch.no_grad()
    def moe_infer(self, x, flat_expert_indices, flat_expert_weights):
        expert_cache = torch.zeros_like(x)
        idxs = flat_expert_indices.argsort()
        tokens_per_expert = flat_expert_indices.bincount().cpu().numpy().cumsum(0)
        token_idxs = idxs // self.config.num_experts_per_tok
        # 当tokens_per_expert = [6, 15, 20, 26]，tokens_per_expert.shape[0]即为专家数量（此时为4）
        # 且token_idxs = [3, 7, 19, 21, 24, 25,  4,  5,  6, 10, 11, 12...] 时
        # 意味token_idxs[:6] -> [3, 7, 19, 21, 24, 25]这6个位置属于专家0处理的token（每个token有可能被多个专家处理，这取决于num_experts_per_tok）
        # 接下来9个位置token_idxs[6:15] -> [4,  5,  6, 10, 11, 12...]属于专家1处理的token...依此类推
        for i, end_idx in enumerate(tokens_per_expert):
            start_idx = 0 if i == 0 else tokens_per_expert[i - 1]
            if start_idx == end_idx:
                continue
            expert = self.experts[i]
            exp_token_idx = token_idxs[start_idx:end_idx]
            expert_tokens = x[exp_token_idx]
            expert_out = expert(expert_tokens).to(expert_cache.dtype)
            expert_out.mul_(flat_expert_weights[idxs[start_idx:end_idx]])
            expert_cache.scatter_add_(0, exp_token_idx.view(-1, 1).repeat(1, x.shape[-1]), expert_out)

        return expert_cache


class MiniMindBlock(nn.Module):
    def __init__(self, layer_id: int, config: MiniMindConfig):
        super().__init__()
        self.num_attention_heads = config.num_attention_heads
        self.hidden_size = config.hidden_size
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.self_attn = Attention(config)

        self.layer_id = layer_id
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.mlp = FeedForward(config) if not config.use_moe else MOEFeedForward(config)

    def forward(self, hidden_states, position_embeddings, past_key_value=None, use_cache=False, attention_mask=None):
        residual = hidden_states
        hidden_states, present_key_value = self.self_attn(
            self.input_layernorm(hidden_states), position_embeddings,
            past_key_value, use_cache, attention_mask
        )
        hidden_states += residual
        hidden_states = hidden_states + self.mlp(self.post_attention_layernorm(hidden_states))
        return hidden_states, present_key_value


class MiniMindModel(nn.Module):
    def __init__(self, config: MiniMindConfig):
        super().__init__()
        self.config = config
        self.vocab_size, self.num_hidden_layers = config.vocab_size, config.num_hidden_layers
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.dropout = nn.Dropout(config.dropout)
        self.layers = nn.ModuleList([MiniMindBlock(l, config) for l in range(self.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

        freqs_cos, freqs_sin = precompute_freqs_cis(dim=config.hidden_size // config.num_attention_heads,
                                                    end=config.max_position_embeddings, rope_base=config.rope_theta,
                                                    rope_scaling=config.rope_scaling)
        self.register_buffer("freqs_cos", freqs_cos, persistent=False)
        self.register_buffer("freqs_sin", freqs_sin, persistent=False)

    def forward(self,
                input_ids: Optional[torch.Tensor] = None,
                attention_mask: Optional[torch.Tensor] = None,
                past_key_values: Optional[List[Tuple[torch.Tensor, torch.Tensor]]] = None,
                use_cache: bool = False,
                **kwargs):
        batch_size, seq_length = input_ids.shape
        if hasattr(past_key_values, 'layers'): past_key_values = None
        past_key_values = past_key_values or [None] * len(self.layers)
        start_pos = past_key_values[0][0].shape[1] if past_key_values[0] is not None else 0

        hidden_states = self.dropout(self.embed_tokens(input_ids))

        position_embeddings = (
            self.freqs_cos[start_pos:start_pos + seq_length],
            self.freqs_sin[start_pos:start_pos + seq_length]
        )

        presents = []
        for layer_idx, (layer, past_key_value) in enumerate(zip(self.layers, past_key_values)):
            hidden_states, present = layer(
                hidden_states,
                position_embeddings,
                past_key_value=past_key_value,
                use_cache=use_cache,
                attention_mask=attention_mask
            )
            presents.append(present)

        hidden_states = self.norm(hidden_states)

        aux_loss = sum(
            layer.mlp.aux_loss
            for layer in self.layers
            if isinstance(layer.mlp, MOEFeedForward)
        )

        return hidden_states, presents, aux_loss


class MiniMindForCausalLM(PreTrainedModel, GenerationMixin):
    config_class = MiniMindConfig

    def __init__(self, config: MiniMindConfig = None):
        self.config = config or MiniMindConfig()
        super().__init__(self.config)
        self.model = MiniMindModel(self.config)
        self.lm_head = nn.Linear(self.config.hidden_size, self.config.vocab_size, bias=False)
        self.model.embed_tokens.weight = self.lm_head.weight
        self.OUT = CausalLMOutputWithPast()

    def forward(self,
                input_ids: Optional[torch.Tensor] = None,
                attention_mask: Optional[torch.Tensor] = None,
                past_key_values: Optional[List[Tuple[torch.Tensor, torch.Tensor]]] = None,
                use_cache: bool = False,
                logits_to_keep: Union[int, torch.Tensor] = 0,
                **args):
        h, past_kvs, aux_loss = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            use_cache=use_cache,
            **args
        )
        slice_indices = slice(-logits_to_keep, None) if isinstance(logits_to_keep, int) else logits_to_keep
        logits = self.lm_head(h[:, slice_indices, :])
        self.OUT.__setitem__('last_hidden_state', h)
        self.OUT.__setitem__('logits', logits)
        self.OUT.__setitem__('aux_loss', aux_loss)
        self.OUT.__setitem__('past_key_values', past_kvs)
        return self.OUT


Writing model/model_minimind.py


In [ ]:
%%writefile pretrain_ddp.py
"""
MiniMind English Pretraining - Production Version
With full DDP support for multi-GPU training

Features:
- Compatible with English tokenizer (<pad>, <unk>, <s>, </s>)
- Full DistributedDataParallel support
- Gradient accumulation
- Mixed precision training (bfloat16/float16)
- Checkpoint resuming
- WandB logging support
"""

import os
import sys
import argparse
import time
import warnings
import torch
import torch.distributed as dist
from contextlib import nullcontext
from torch import optim, nn
from torch.nn.parallel import DistributedDataParallel
from torch.utils.data import DataLoader, DistributedSampler
from pathlib import Path
import json

# Add parent directory to path for imports
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

warnings.filterwarnings('ignore')

# ============================================================================
# Import MiniMind Model (assumes model/model_minimind.py exists)
# ============================================================================
try:
    from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
    USE_MINIMIND_MODEL = True
except ImportError:
    print("⚠️  Could not import MiniMind model, will use built-in simple model")
    USE_MINIMIND_MODEL = False

# ============================================================================
# Tokenizer Wrapper
# ============================================================================
class EnglishTokenizer:
    """Wrapper for English BPE tokenizer with correct special tokens"""
    def __init__(self, tokenizer_path='model/tokenizer.json'):
        from tokenizers import Tokenizer

        print(f"Loading tokenizer from: {tokenizer_path}")
        self.tokenizer = Tokenizer.from_file(tokenizer_path)
        self.vocab_size = self.tokenizer.get_vocab_size()

        vocab = self.tokenizer.get_vocab()

        # ✅ FIXED: Use correct special token names
        self.pad_id = vocab.get('<pad>', 0)
        self.unk_id = vocab.get('<unk>', 1)
        self.bos_id = vocab.get('<s>', 2)
        self.eos_id = vocab.get('</s>', 3)

        # Verify tokens exist
        if self.pad_id == self.bos_id == self.eos_id == 0:
            raise ValueError(
                "❌ ERROR: Special tokens not found! "
                "Please retrain tokenizer with correct tokens: <pad>, <unk>, <s>, </s>"
            )

        print(f"✅ Tokenizer loaded:")
        print(f"   Vocab size: {self.vocab_size:,}")
        print(f"   PAD: {self.pad_id} (<pad>)")
        print(f"   UNK: {self.unk_id} (<unk>)")
        print(f"   BOS: {self.bos_id} (<s>)")
        print(f"   EOS: {self.eos_id} (</s>)")

    def encode(self, text, add_special_tokens=True):
        """Encode text to token IDs"""
        encoding = self.tokenizer.encode(text)
        ids = encoding.ids
        if add_special_tokens:
            ids = [self.bos_id] + ids + [self.eos_id]
        return ids

    def decode(self, ids):
        """Decode token IDs to text"""
        if isinstance(ids, torch.Tensor):
            ids = ids.tolist()
        # Remove special tokens
        ids = [id for id in ids if id not in [self.pad_id, self.bos_id, self.eos_id]]
        return self.tokenizer.decode(ids)

    def __len__(self):
        return self.vocab_size

# ============================================================================
# Dataset
# ============================================================================
class PretrainDataset(torch.utils.data.Dataset):
    """Pretraining dataset from JSONL file"""
    def __init__(self, data_path, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length

        print(f"Loading data from {data_path}...")
        self.samples = []

        with open(data_path, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line.strip())
                if data.get('text'):
                    self.samples.append(data['text'])

        print(f"✅ Loaded {len(self.samples):,} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text = self.samples[idx]

        # Encode with BOS and EOS
        tokens = self.tokenizer.encode(text, add_special_tokens=True)

        # Truncate if too long
        if len(tokens) > self.max_length + 1:
            tokens = tokens[:self.max_length + 1]

        # Convert to tensor
        tokens_tensor = torch.tensor(tokens, dtype=torch.long)

        # Pad if needed
        if len(tokens_tensor) < self.max_length + 1:
            padding = torch.full(
                (self.max_length + 1 - len(tokens_tensor),),
                self.tokenizer.pad_id,
                dtype=torch.long
            )
            tokens_tensor = torch.cat([tokens_tensor, padding])

        # Create input (all but last) and target (all but first)
        x = tokens_tensor[:-1]
        y = tokens_tensor[1:]

        # Create loss mask (1 for real tokens, 0 for padding)
        loss_mask = (y != self.tokenizer.pad_id).float()

        return x, y, loss_mask

# ============================================================================
# Training Utilities
# ============================================================================

def init_distributed_mode():
    """Initialize distributed training"""
    if 'RANK' in os.environ and 'WORLD_SIZE' in os.environ:
        rank = int(os.environ["RANK"])
        world_size = int(os.environ['WORLD_SIZE'])
        local_rank = int(os.environ['LOCAL_RANK'])
    else:
        print('Not using distributed mode')
        return None

    torch.cuda.set_device(local_rank)
    dist.init_process_group(
        backend='nccl',
        init_method='env://',
        world_size=world_size,
        rank=rank
    )
    dist.barrier()
    return local_rank

def is_main_process():
    """Check if current process is main"""
    return not dist.is_initialized() or dist.get_rank() == 0

def setup_seed(seed):
    """Set random seed for reproducibility"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def get_lr(step, total_steps, learning_rate, warmup_steps=2000, min_lr=None):
    """Learning rate schedule: warmup + cosine decay"""
    if min_lr is None:
        min_lr = learning_rate * 0.1

    # Warmup
    if step < warmup_steps:
        return learning_rate * step / warmup_steps

    # Cosine decay
    if step > total_steps:
        return min_lr

    decay_ratio = (step - warmup_steps) / (total_steps - warmup_steps)
    coeff = 0.5 * (1.0 + torch.cos(torch.tensor(decay_ratio * 3.14159)))
    return min_lr + coeff * (learning_rate - min_lr)

def Logger(msg):
    """Print log message with timestamp"""
    if is_main_process():
        print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

class SkipBatchSampler:
    """Batch sampler that skips first N steps (for checkpoint resuming)"""
    def __init__(self, sampler, batch_size, skip_steps):
        self.sampler = sampler
        self.batch_size = batch_size
        self.skip_steps = skip_steps

    def __iter__(self):
        batch = []
        skip_count = 0

        for idx in self.sampler:
            if skip_count < self.skip_steps * self.batch_size:
                skip_count += 1
                continue

            batch.append(idx)
            if len(batch) == self.batch_size:
                yield batch
                batch = []

        if len(batch) > 0:
            yield batch

    def __len__(self):
        total_batches = (len(self.sampler) + self.batch_size - 1) // self.batch_size
        return max(0, total_batches - self.skip_steps)

def save_checkpoint(args, model, optimizer, scaler, epoch, step, lm_config):
    """Save training checkpoint"""
    if not is_main_process():
        return

    checkpoint_dir = Path(args.save_dir) / 'checkpoints'
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    # Get model state dict
    if isinstance(model, torch.nn.parallel.DistributedDataParallel):
        state_dict = model.module.state_dict()
    else:
        state_dict = model.state_dict()

    # Save checkpoint
    checkpoint = {
        'epoch': epoch,
        'step': step,
        'model': state_dict,
        'optimizer': optimizer.state_dict(),
        'scaler': scaler.state_dict(),
        'config': vars(lm_config),
        'args': vars(args),
    }

    ckpt_path = checkpoint_dir / f'checkpoint_epoch{epoch}_step{step}.pt'
    torch.save(checkpoint, ckpt_path)

    # Also save as latest
    latest_path = checkpoint_dir / 'checkpoint_latest.pt'
    torch.save(checkpoint, latest_path)

    Logger(f"💾 Checkpoint saved: {ckpt_path}")

def load_checkpoint(checkpoint_path, model, optimizer, scaler):
    """Load training checkpoint"""
    if not os.path.exists(checkpoint_path):
        return None

    Logger(f"Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location='cpu')

    # Load states
    if isinstance(model, torch.nn.parallel.DistributedDataParallel):
        model.module.load_state_dict(checkpoint['model'])
    else:
        model.load_state_dict(checkpoint['model'])

    optimizer.load_state_dict(checkpoint['optimizer'])
    scaler.load_state_dict(checkpoint['scaler'])

    Logger(f"✅ Resumed from epoch {checkpoint['epoch']}, step {checkpoint['step']}")

    return checkpoint

# ============================================================================
# Training Loop
# ============================================================================

def train_epoch(epoch, loader, iters, model, optimizer, scaler, args, lm_config, autocast_ctx, start_step=0, wandb=None):
    """Train for one epoch"""
    model.train()
    loss_fct = nn.CrossEntropyLoss(reduction='none')
    start_time = time.time()
    total_steps = args.epochs * iters

    for step, (X, Y, loss_mask) in enumerate(loader, start=start_step):
        X = X.to(args.device)
        Y = Y.to(args.device)
        loss_mask = loss_mask.to(args.device)

        # Calculate learning rate
        current_step = epoch * iters + step
        lr = get_lr(
            current_step,
            total_steps,
            args.learning_rate,
            args.warmup_steps,
            args.min_lr
        )

        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # Forward pass with mixed precision
        with autocast_ctx:
            if USE_MINIMIND_MODEL:
                res = model(X)
                logits = res.logits
                aux_loss = res.aux_loss if hasattr(res, 'aux_loss') else 0
            else:
                logits = model(X)
                aux_loss = 0

            # Calculate loss
            loss = loss_fct(
                logits.reshape(-1, logits.size(-1)),
                Y.reshape(-1)
            ).view(Y.size())

            loss = (loss * loss_mask).sum() / loss_mask.sum()
            loss = loss + aux_loss
            loss = loss / args.accumulation_steps

        # Backward pass
        scaler.scale(loss).backward()

        # Optimizer step with gradient accumulation
        if (step + 1) % args.accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        # Logging
        if step % args.log_interval == 0:
            spend_time = time.time() - start_time
            current_loss = loss.item() * args.accumulation_steps
            eta_min = spend_time / (step + 1) * iters // 60 - spend_time // 60

            Logger(
                f'Epoch:[{epoch+1}/{args.epochs}]({step}/{iters}) '
                f'loss:{current_loss:.6f} lr:{lr:.2e} '
                f'eta:{eta_min:.0f}min'
            )

            if wandb:
                wandb.log({
                    "loss": current_loss,
                    "lr": lr,
                    "epoch": epoch + 1,
                    "step": current_step
                })

        # Save checkpoint
        if (step % args.save_interval == 0 and step > 0) or step == iters - 1:
            save_checkpoint(args, model, optimizer, scaler, epoch, step, lm_config)

# ============================================================================
# Main Training Function
# ============================================================================

def main(args):
    # ========== 1. Initialize distributed training ==========
    local_rank = init_distributed_mode()
    if dist.is_initialized():
        args.device = f"cuda:{local_rank}"

    setup_seed(42 + (dist.get_rank() if dist.is_initialized() else 0))

    Logger("="*70)
    Logger("MiniMind English Pretraining")
    Logger("="*70)
    Logger(f"Device: {args.device}")
    Logger(f"Distributed: {dist.is_initialized()}")
    if dist.is_initialized():
        Logger(f"World size: {dist.get_world_size()}")
        Logger(f"Rank: {dist.get_rank()}")
    Logger("="*70)

    # ========== 2. Create output directories ==========
    os.makedirs(args.save_dir, exist_ok=True)
    os.makedirs(os.path.join(args.save_dir, 'checkpoints'), exist_ok=True)

    # ========== 3. Load tokenizer ==========
    Logger("Loading tokenizer...")
    tokenizer = EnglishTokenizer(args.tokenizer_path)

    # ========== 4. Create model configuration ==========
    if USE_MINIMIND_MODEL:
        Logger("Using MiniMind model architecture")
        lm_config = MiniMindConfig(
            vocab_size=tokenizer.vocab_size,
            hidden_size=args.hidden_size,
            num_hidden_layers=args.num_hidden_layers,
            num_attention_heads=args.num_attention_heads,
            num_key_value_heads=args.num_key_value_heads,
            max_position_embeddings=args.max_seq_len,
            dropout=args.dropout,
            use_moe=bool(args.use_moe),
            bos_token_id=tokenizer.bos_id,
            eos_token_id=tokenizer.eos_id,
        )
    else:
        Logger("⚠️  Using simplified model (MiniMind not available)")
        lm_config = None

    # ========== 5. Initialize model ==========
    Logger("Initializing model...")
    if USE_MINIMIND_MODEL:
        model = MiniMindForCausalLM(lm_config)
    else:
        # Fallback to simple transformer if MiniMind not available
        raise ImportError(
            "MiniMind model not found. Please ensure model/model_minimind.py exists"
        )

    model = model.to(args.device)

    n_params = sum(p.numel() for p in model.parameters())
    Logger(f"Model parameters: {n_params/1e6:.2f}M")

    # ========== 6. Setup mixed precision ==========
    device_type = "cuda" if "cuda" in args.device else "cpu"
    dtype = torch.bfloat16 if args.dtype == "bfloat16" else torch.float16
    autocast_ctx = (
        nullcontext() if device_type == "cpu"
        else torch.cuda.amp.autocast(dtype=dtype)
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(args.dtype == 'float16'))

    # ========== 7. Create dataset and dataloader ==========
    Logger("Creating dataset...")
    train_ds = PretrainDataset(args.data_path, tokenizer, args.max_seq_len)

    train_sampler = None
    if dist.is_initialized():
        train_sampler = DistributedSampler(
            train_ds,
            num_replicas=dist.get_world_size(),
            rank=dist.get_rank(),
            shuffle=True
        )

    # ========== 8. Create optimizer ==========
    optimizer = optim.AdamW(
        model.parameters(),
        lr=args.learning_rate,
        weight_decay=args.weight_decay,
        betas=(0.9, 0.95)
    )

    # ========== 9. Load checkpoint if resuming ==========
    start_epoch = 0
    start_step = 0

    if args.resume:
        checkpoint_path = os.path.join(args.save_dir, 'checkpoints', 'checkpoint_latest.pt')
        checkpoint = load_checkpoint(checkpoint_path, model, optimizer, scaler)
        if checkpoint:
            start_epoch = checkpoint['epoch']
            start_step = checkpoint.get('step', 0)

    # ========== 10. Wrap model with DDP ==========
    if dist.is_initialized():
        Logger("Wrapping model with DistributedDataParallel...")
        # Ignore position embeddings for DDP
        if hasattr(model, 'model'):
            model.model._ddp_params_and_buffers_to_ignore = {"freqs_cos", "freqs_sin"}
        model = DistributedDataParallel(
            model,
            device_ids=[local_rank],
            output_device=local_rank
        )

    # ========== 11. Setup WandB ==========
    wandb = None
    if args.use_wandb and is_main_process():
        try:
            import wandb
            wandb.init(
                project=args.wandb_project,
                name=f"MiniMind-English-{args.hidden_size}",
                config=vars(args)
            )
            Logger("✅ WandB initialized")
        except ImportError:
            Logger("⚠️  WandB not available, skipping")
            wandb = None

    # ========== 12. Start training ==========
    Logger("\n" + "="*70)
    Logger("STARTING TRAINING")
    Logger("="*70)
    Logger(f"Total epochs: {args.epochs}")
    Logger(f"Batch size per GPU: {args.batch_size}")
    if dist.is_initialized():
        Logger(f"Global batch size: {args.batch_size * dist.get_world_size()}")
    Logger(f"Gradient accumulation steps: {args.accumulation_steps}")
    Logger(f"Effective batch size: {args.batch_size * args.accumulation_steps}")
    Logger("="*70 + "\n")

    for epoch in range(start_epoch, args.epochs):
        if train_sampler:
            train_sampler.set_epoch(epoch)

        # Handle checkpoint resuming for first epoch
        if epoch == start_epoch and start_step > 0:
            Logger(f"Resuming epoch {epoch+1} from step {start_step+1}")
            batch_sampler = SkipBatchSampler(
                train_sampler or range(len(train_ds)),
                args.batch_size,
                start_step + 1
            )
            loader = DataLoader(
                train_ds,
                batch_sampler=batch_sampler,
                num_workers=args.num_workers,
                pin_memory=True
            )
            iters = len(loader) + start_step + 1
            train_epoch(
                epoch, loader, iters, model, optimizer, scaler,
                args, lm_config, autocast_ctx, start_step, wandb
            )
        else:
            # Normal training
            loader = DataLoader(
                train_ds,
                batch_size=args.batch_size,
                shuffle=(train_sampler is None),
                sampler=train_sampler,
                num_workers=args.num_workers,
                pin_memory=True,
                drop_last=True
            )
            iters = len(loader)
            train_epoch(
                epoch, loader, iters, model, optimizer, scaler,
                args, lm_config, autocast_ctx, 0, wandb
            )

    Logger("\n" + "="*70)
    Logger("🎉 TRAINING COMPLETE!")
    Logger("="*70)

    if wandb:
        wandb.finish()

    if dist.is_initialized():
        dist.destroy_process_group()

# ============================================================================
# Entry Point
# ============================================================================

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="MiniMind English Pretraining")

    # Paths
    parser.add_argument("--data_path", type=str, default="data/processed/pretrain.jsonl",
                        help="Path to pretraining data (JSONL format)")
    parser.add_argument("--tokenizer_path", type=str, default="model/tokenizer.json",
                        help="Path to tokenizer.json")
    parser.add_argument("--save_dir", type=str, default="out",
                        help="Directory to save checkpoints")

    # Model architecture
    parser.add_argument("--hidden_size", type=int, default=768,
                        help="Hidden dimension size")
    parser.add_argument("--num_hidden_layers", type=int, default=16,
                        help="Number of transformer layers")
    parser.add_argument("--num_attention_heads", type=int, default=16,
                        help="Number of attention heads")
    parser.add_argument("--num_key_value_heads", type=int, default=8,
                        help="Number of KV heads (for GQA)")
    parser.add_argument("--max_seq_len", type=int, default=512,
                        help="Maximum sequence length")
    parser.add_argument("--dropout", type=float, default=0.0,
                        help="Dropout rate")
    parser.add_argument("--use_moe", type=int, default=0, choices=[0, 1],
                        help="Use Mixture of Experts (0=no, 1=yes)")

    # Training hyperparameters
    parser.add_argument("--epochs", type=int, default=1,
                        help="Number of training epochs")
    parser.add_argument("--batch_size", type=int, default=32,
                        help="Batch size per GPU")
    parser.add_argument("--learning_rate", type=float, default=3e-4,
                        help="Initial learning rate")
    parser.add_argument("--min_lr", type=float, default=3e-5,
                        help="Minimum learning rate")
    parser.add_argument("--warmup_steps", type=int, default=2000,
                        help="Number of warmup steps")
    parser.add_argument("--weight_decay", type=float, default=0.1,
                        help="Weight decay")
    parser.add_argument("--grad_clip", type=float, default=1.0,
                        help="Gradient clipping threshold")
    parser.add_argument("--accumulation_steps", type=int, default=1,
                        help="Gradient accumulation steps")

    # Training configuration
    parser.add_argument("--dtype", type=str, default="bfloat16",
                        choices=["float16", "bfloat16", "float32"],
                        help="Training precision")
    parser.add_argument("--num_workers", type=int, default=4,
                        help="Number of dataloader workers")
    parser.add_argument("--device", type=str, default="cuda:0",
                        help="Training device")

    # Logging and checkpointing
    parser.add_argument("--log_interval", type=int, default=100,
                        help="Steps between logging")
    parser.add_argument("--save_interval", type=int, default=5000,
                        help="Steps between checkpoints")
    parser.add_argument("--resume", action="store_true",
                        help="Resume from latest checkpoint")

    # WandB
    parser.add_argument("--use_wandb", action="store_true",
                        help="Use Weights & Biases logging")
    parser.add_argument("--wandb_project", type=str, default="MiniMind-English",
                        help="WandB project name")

    args = parser.parse_args()

    main(args)


Writing pretrain_ddp.py


In [ ]:
#the following configurations was set for signle A100 GPU.
!python pretrain_ddp.py \
    --data_path data/processed/pretrain.jsonl \
    --tokenizer_path model/tokenizer.json \
    --save_dir out \
    --hidden_size 512 \
    --num_hidden_layers 8 \
    --num_attention_heads 16 \
    --num_key_value_heads 8 \
    --max_seq_len 512 \
    --dropout 0.0 \
    --use_moe 0 \
    --epochs 1 \
    --batch_size 64 \   #reduce to 8 if using T4 GPU
    --learning_rate 1e-4 \
    --min_lr 1e-5 \
    --warmup_steps 5000 \
    --weight_decay 0.1 \
    --grad_clip 0.5 \
    --accumulation_steps 1 \
    --dtype bfloat16 \
    --num_workers 4 \
    --log_interval 500 \
    --save_interval 5000

2025-11-09 18:17:13.473039: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-09 18:17:13.490742: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762712233.512238   14177 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762712233.518858   14177 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762712233.535323   14177 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

If you can access multiple GPUs, for example, two A100OR H100, you can use:



In [ ]:
torchrun --nproc_per_node 2 \
    pretrain_ddp.py  \
    --data_path data/processed/pretrain.jsonl \
    --tokenizer_path model/tokenizer.json \
    --save_dir out \
    --hidden_size 768 \
    --num_hidden_layers 16 \
    --num_attention_heads 16 \
    --num_key_value_heads 8 \
    --max_seq_len 512 \
    --dropout 0.0 \
    --epochs 1 \
    --batch_size 32 \
    --learning_rate 1e-4 \
    --min_lr 1e-5 \
    --warmup_steps 5000 \
    --weight_decay 0.1 \
    --grad_clip 0.5 \
    --accumulation_steps 1 \
    --dtype bfloat16 \
    --num_workers 4 \
    --log_interval 500 \
    --save_interval 5000

or if you can access HPC server, you can reference the running scripts I used:


In [ ]:
**LSF system**
"run_pretrain.sh"
```
#BSUB -n 2
#BSUB -W 1200
#BSUB -q gpu
#BSUB -R "select[h200||h100||a100]"
#BSUB -gpu "num=2"
#BSUB -o out.%J
#BSUB -e err.%J
source ~/.bashrc

conda activate /usr/local/usrapps/sediment/minimind

# Optional: print some useful info
echo "Job started at $(date)"
echo "Working directory: $(pwd)"
echo "Job ID: $LSB_JOBID"
echo "Hosts: $LSB_HOSTS"
echo "GPUs: $CUDA_VISIBLE_DEVICES"

# Run your work
cd /share/USER_ID/gpt3    #change to yours

# Use a random port to avoid conflicts
MASTER_PORT=$((29500 + RANDOM % 1000))

torchrun --nproc_per_node 2 \
    --master_port $MASTER_PORT \
    pretrain_ddp.py  \
    --data_path data/processed/pretrain.jsonl \
    --tokenizer_path model/tokenizer.json \
    --save_dir out \
    --hidden_size 768 \
    --num_hidden_layers 16 \
    --num_attention_heads 16 \
    --num_key_value_heads 8 \
    --max_seq_len 512 \
    --dropout 0.0 \
    --epochs 1 \
    --batch_size 32 \
    --learning_rate 1e-4 \
    --min_lr 1e-5 \
    --warmup_steps 5000 \
    --weight_decay 0.1 \
    --grad_clip 0.5 \
    --accumulation_steps 1 \
    --dtype bfloat16 \
    --num_workers 4 \
    --log_interval 500 \
    --save_interval 5000

conda deactivate
echo "Job done at $(date)"

In [ ]:
**SLURM system**
```
#!/bin/bash
#SBATCH --job-name=two_gpu_job
#SBATCH --partition=gpu
#SBATCH --constraint=a100|h100|h200   #depend on your system
#SBATCH --gres=gpu:2
#SBATCH --mem=160G
#SBATCH --cpus-per-task=8
#SBATCH --time=24:00:00          # adjust if needed
#SBATCH --output=/work/pliu1/gpt3/slurm-%j.out
#SBATCH --error=/work/pliu1/gpt3/slurm-%j.err

# Activate your conda
source ~/miniforge3/etc/profile.d/conda.sh
conda activate minimind

# Optional: print some useful info
echo "Job started at $(date)"
echo "Working directory: $(pwd)"
echo "Alloc nodes: $SLURM_NODELIST"
echo "GPUs: $CUDA_VISIBLE_DEVICES"

# Run your work
cd /work/USER_ID/gpt3

#run under 2 GPUs
torchrun --nproc_per_node 2 pretrain_ddp.py \
    --data_path data/processed/pretrain.jsonl \
    --tokenizer_path model/tokenizer.json \
    --save_dir out \
    --hidden_size 768 \
    --num_hidden_layers 16 \
    --num_attention_heads 16 \
    --num_key_value_heads 8 \
    --max_seq_len 512 \
    --dropout 0.0 \
    --use_moe 0 \
    --epochs 2 \
    --batch_size 128 \
    --learning_rate 1e-4 \
    --min_lr 1e-5 \
    --warmup_steps 5000 \
    --weight_decay 0.1 \
    --grad_clip 0.5 \
    --accumulation_steps 1 \
    --dtype bfloat16 \
    --num_workers 4 \
    --log_interval 500 \
    --save_interval 5000

echo "Job done at $(date)"
```

In [ ]:
%%writefile test_pretrained.py
"""
Quick Inference Test for Pretrained Models - FIXED VERSION

CRITICAL FIX: Adds BOS token during inference to match training!

Usage:
    python test_pretrained_FIXED.py --checkpoint out/checkpoints/checkpoint_latest.pt
"""

import argparse
import torch
import sys
import os
from transformers import AutoTokenizer

# Add parent directory to path
sys.path.append(os.path.dirname(os.path.abspath(__file__)))

from model.model_minimind import MiniMindConfig, MiniMindForCausalLM

def load_model(checkpoint_path, tokenizer_path, hidden_size, num_layers,
               num_attention_heads, num_kv_heads, device):
    """Load model"""
    print(f"\n{'='*70}")
    print("Loading Model...")
    print(f"{'='*70}\n")

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    print(f"✅ Tokenizer loaded ({len(tokenizer):,} tokens)")
    print(f"   BOS token: {tokenizer.bos_token} (ID: {tokenizer.bos_token_id})")
    print(f"   EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
    print(f"   PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")

    # Create model
    config = MiniMindConfig(
        vocab_size=len(tokenizer),
        hidden_size=hidden_size,
        num_hidden_layers=num_layers,
        num_attention_heads=num_attention_heads,
        num_key_value_heads=num_kv_heads,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    model = MiniMindForCausalLM(config)

    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['model'] if 'model' in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=True)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"✅ Model loaded ({n_params/1e6:.2f}M parameters)")
    print(f"✅ Device: {device}\n")

    return model.to(device).eval(), tokenizer

def generate(model, tokenizer, prompt, max_tokens=100, temperature=0.8, device='cuda'):
    """Generate text completion - FIXED to add BOS token!"""

    # ⭐ CRITICAL FIX: Add BOS token manually
    # The model was trained with BOS at position 0, so we MUST add it during inference!

    # Tokenize WITHOUT adding special tokens (we'll add BOS manually)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)

    # Prepend BOS token to match training format
    bos_token = torch.tensor([[tokenizer.bos_token_id]], dtype=torch.long, device=device)
    input_ids_with_bos = torch.cat([bos_token, inputs["input_ids"]], dim=1)

    # Update attention mask
    attention_mask_with_bos = torch.cat([
        torch.ones((1, 1), dtype=torch.long, device=device),
        inputs["attention_mask"]
    ], dim=1)

    print(f"\n🔍 Debug - Input IDs:")
    print(f"   First 10 tokens: {input_ids_with_bos[0, :10].tolist()}")
    print(f"   First token is BOS? {input_ids_with_bos[0, 0].item() == tokenizer.bos_token_id}")

    # CRITICAL: Ensure all special tokens are set correctly
    gen_kwargs = {
        "input_ids": input_ids_with_bos,
        "attention_mask": attention_mask_with_bos,
        "max_new_tokens": max_tokens,
        "pad_token_id": tokenizer.pad_token_id,
        "bos_token_id": tokenizer.bos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    # Use appropriate decoding strategy
    if temperature > 0:
        gen_kwargs.update({
            "do_sample": True,
            "temperature": temperature,
            "top_p": 0.9,
            "top_k": 50,
            "repetition_penalty": 1.2,  # IMPORTANT: Prevent repetition
            "no_repeat_ngram_size": 3,  # Also helps
        })
    else:
        gen_kwargs["do_sample"] = False

    with torch.no_grad():
        outputs = model.generate(**gen_kwargs)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", type=str, required=True)
    parser.add_argument("--tokenizer_path", type=str, default="model")
    parser.add_argument("--hidden_size", type=int, default=768)
    parser.add_argument("--num_layers", type=int, default=16)
    parser.add_argument("--num_attention_heads", type=int, default=16)
    parser.add_argument("--num_kv_heads", type=int, default=8)
    parser.add_argument("--device", type=str,
                        default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--temperature", type=float, default=0.8)
    parser.add_argument("--max_tokens", type=int, default=200)
    args = parser.parse_args()

    # Load model
    model, tokenizer = load_model(
        args.checkpoint,
        args.tokenizer_path,
        args.hidden_size,
        args.num_layers,
        args.num_attention_heads,
        args.num_kv_heads,
        args.device
    )

    # Test prompts
    test_prompts = [
        "The decision left CNN Chief Medical Correspondent Sanjay Gupta",
        "The capital of France is",
        "Once upon a time,",
        "Artificial intelligence is",
        "The human brain",
        "In the year 2050,",
    ]

    print(f"{'='*70}")
    print("Text Completion Test (FIXED VERSION)")
    print(f"{'='*70}")
    print("⭐ Now correctly adds BOS token to match training!")
    print(f"{'='*70}\n")

    for i, prompt in enumerate(test_prompts, 1):
        print(f"{i}. Prompt: \"{prompt}\"")
        completion = generate(
            model, tokenizer, prompt,
            max_tokens=args.max_tokens,
            temperature=args.temperature,
            device=args.device
        )
        print(f"   Output: {completion}\n")

    print(f"{'='*70}")
    print("✅ Test Complete!")
    print(f"{'='*70}\n")

    # Interactive mode
    print("Enter prompts to continue testing (or 'quit' to exit):\n")

    while True:
        try:
            prompt = input("Prompt: ").strip()
            if not prompt or prompt.lower() in ['quit', 'exit', 'q']:
                break

            completion = generate(
                model, tokenizer, prompt,
                max_tokens=args.max_tokens,
                temperature=args.temperature,
                device=args.device
            )
            print(f"\n{completion}\n")

        except (KeyboardInterrupt, EOFError):
            break

    print("\n👋 Goodbye!")

if __name__ == "__main__":
    main()


Writing test_pretrain.py


In [ ]:
pwd

'/content/drive/MyDrive/gpt3'

In [ ]:
!python test_pretrained.py --checkpoint out/checkpoints/checkpoint_latest.pt --hidden_size 512 --num_layers 8


2025-11-10 00:39:47.782501: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-10 00:39:47.801475: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762735187.824400  109201 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762735187.831220  109201 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762735187.848661  109201 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 